# Phase 8 — Predictive Model Development

## Objective

Develop and compare classification models that predict whether an eligible customer will make at least one qualifying purchase within the 90-day outcome window following a monthly snapshot date.

## Modeling Approach

The modeling process uses the frozen customer purchase-propensity dataset constructed in Phase 7. Model development and selection will use the chronological training and validation partitions. The test partition will remain untouched until final evaluation.

The initial modeling approach will compare Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting classifiers.

## Evaluation

Models will be assessed using ROC-AUC, PR-AUC, precision, recall, F1-score, and confusion matrices. Model selection will consider predictive performance, class-specific errors, interpretability, and business relevance.

## Scope

This phase covers model development, comparison, and selection. Final test-set evaluation and model explainability will be addressed in Phase 9.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

DATA_PATHS = [
    Path("data/processed/customer_purchase_propensity_dataset.parquet"),
    Path("../data/processed/customer_purchase_propensity_dataset.parquet"),
]

DATA_PATH = next(
    (path for path in DATA_PATHS if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate customer_purchase_propensity_dataset.parquet."
    )

modeling_df = pd.read_parquet(DATA_PATH)

print("Scikit-learn version:", sklearn.__version__)
print("Dataset path:", DATA_PATH)
print("Dataset shape:", modeling_df.shape)

print("\nColumns:")
print(modeling_df.columns.tolist())

print("\nPartition counts:")
print(modeling_df["partition"].value_counts().sort_index())

print("\nTarget distribution:")
print(modeling_df["repeat_purchase_90d"].value_counts().sort_index())

print("\nMissing values:")
print(modeling_df.isna().sum())

Scikit-learn version: 1.9.1
Dataset path: ..\data\processed\customer_purchase_propensity_dataset.parquet
Dataset shape: (47933, 9)

Columns:
['Customer ID', 'snapshot_date', 'recency_days', 'log1p_frequency', 'log1p_monetary', 'log1p_unique_products', 'log1p_cancellation_value', 'repeat_purchase_90d', 'partition']

Partition counts:
partition
embargo       15295
test           5534
train         17119
validation     9985
Name: count, dtype: int64

Target distribution:
repeat_purchase_90d
0    24400
1    23533
Name: count, dtype: int64

Missing values:
Customer ID                 0
snapshot_date               0
recency_days                0
log1p_frequency             0
log1p_monetary              0
log1p_unique_products       0
log1p_cancellation_value    0
repeat_purchase_90d         0
partition                   0
dtype: int64


In [3]:
# Define the predictors and target
feature_columns = [
    "recency_days",
    "log1p_frequency",
    "log1p_monetary",
    "log1p_unique_products",
    "log1p_cancellation_value",
]

target_column = "repeat_purchase_90d"

# Separate chronological partitions
train_df = modeling_df.loc[
    modeling_df["partition"].eq("train")
].copy()

validation_df = modeling_df.loc[
    modeling_df["partition"].eq("validation")
].copy()

test_df = modeling_df.loc[
    modeling_df["partition"].eq("test")
].copy()

# Create feature matrices and target vectors
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_validation = validation_df[feature_columns]
y_validation = validation_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

# Verify dimensions and target distributions
for name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_validation, y_validation),
    ("Test", X_test, y_test),
]:
    print(f"\n{name}")
    print("Features:", X.shape)
    print("Target:", y.shape)
    print("Positive rate:", round(y.mean(), 4))
    print("Target counts:")
    print(y.value_counts().sort_index())

print("\nFeature columns:", X_train.columns.tolist())
print("\nEmbargo observations excluded:", len(modeling_df) - len(train_df) - len(validation_df) - len(test_df))


Train
Features: (17119, 5)
Target: (17119,)
Positive rate: 0.5451
Target counts:
repeat_purchase_90d
0    7787
1    9332
Name: count, dtype: int64

Validation
Features: (9985, 5)
Target: (9985,)
Positive rate: 0.4196
Target counts:
repeat_purchase_90d
0    5795
1    4190
Name: count, dtype: int64

Test
Features: (5534, 5)
Target: (5534,)
Positive rate: 0.5819
Target counts:
repeat_purchase_90d
0    2314
1    3220
Name: count, dtype: int64

Feature columns: ['recency_days', 'log1p_frequency', 'log1p_monetary', 'log1p_unique_products', 'log1p_cancellation_value']

Embargo observations excluded: 15295


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Define the baseline model
logistic_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train using training data only
logistic_baseline.fit(X_train, y_train)

print("Logistic Regression baseline trained successfully.")

print("\nModel parameters:")
print(logistic_baseline.named_steps["classifier"].get_params())

Logistic Regression baseline trained successfully.

Model parameters:
{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


In [5]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

def evaluate_classifier(model, X, y, dataset_name):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results = {
        "dataset": dataset_name,
        "roc_auc": roc_auc_score(y, probabilities),
        "pr_auc": average_precision_score(y, probabilities),
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
    }

    print(f"\n{dataset_name} results")
    for metric, value in results.items():
        if metric != "dataset":
            print(f"{metric}: {value:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y, predictions))

    return results


train_results = evaluate_classifier(
    logistic_baseline,
    X_train,
    y_train,
    "Training",
)

validation_results = evaluate_classifier(
    logistic_baseline,
    X_validation,
    y_validation,
    "Validation",
)

logistic_results_df = pd.DataFrame([
    train_results,
    validation_results,
])

print("\nPerformance comparison:")
display(logistic_results_df.set_index("dataset").round(4))


Training results
roc_auc: 0.7267
pr_auc: 0.7731
accuracy: 0.6662
precision: 0.7014
recall: 0.6749
f1: 0.6879

Confusion matrix:
[[5106 2681]
 [3034 6298]]

Validation results
roc_auc: 0.7468
pr_auc: 0.7187
accuracy: 0.6646
precision: 0.5812
recall: 0.7181
f1: 0.6425

Confusion matrix:
[[3627 2168]
 [1181 3009]]

Performance comparison:


,roc_auc,pr_auc,accuracy,precision,recall,f1
dataset,,,,,,
Training,0.7267,0.7731,0.6662,0.7014,0.6749,0.6879
Validation,0.7468,0.7187,0.6646,0.5812,0.7181,0.6425


In [6]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42
)

decision_tree.fit(X_train, y_train)

print("Decision Tree trained successfully.")
print("Maximum depth:", decision_tree.get_depth())
print("Number of leaves:", decision_tree.get_n_leaves())

Decision Tree trained successfully.
Maximum depth: 5
Number of leaves: 31


In [7]:
tree_train_results = evaluate_classifier(
    decision_tree,
    X_train,
    y_train,
    "Decision Tree - Training",
)

tree_validation_results = evaluate_classifier(
    decision_tree,
    X_validation,
    y_validation,
    "Decision Tree - Validation",
)


Decision Tree - Training results
roc_auc: 0.7363
pr_auc: 0.7647
accuracy: 0.6751
precision: 0.7026
recall: 0.7006
f1: 0.7016

Confusion matrix:
[[5019 2768]
 [2794 6538]]

Decision Tree - Validation results
roc_auc: 0.7338
pr_auc: 0.6885
accuracy: 0.6488
precision: 0.5641
recall: 0.7172
f1: 0.6315

Confusion matrix:
[[3473 2322]
 [1185 3005]]


In [8]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

print("Random Forest trained successfully.")
print("Number of trees:", len(random_forest.estimators_))

Random Forest trained successfully.
Number of trees: 300


In [9]:
rf_train_results = evaluate_classifier(
    random_forest,
    X_train,
    y_train,
    "Random Forest - Training",
)

rf_validation_results = evaluate_classifier(
    random_forest,
    X_validation,
    y_validation,
    "Random Forest - Validation",
)


Random Forest - Training results
roc_auc: 0.7890
pr_auc: 0.8283
accuracy: 0.7078
precision: 0.7404
recall: 0.7144
f1: 0.7272

Confusion matrix:
[[5449 2338]
 [2665 6667]]

Random Forest - Validation results
roc_auc: 0.7398
pr_auc: 0.7121
accuracy: 0.6554
precision: 0.5709
recall: 0.7196
f1: 0.6367

Confusion matrix:
[[3529 2266]
 [1175 3015]]


In [10]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=30,
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

print("Gradient Boosting trained successfully.")
print("Number of estimators:", len(gradient_boosting.estimators_))

Gradient Boosting trained successfully.
Number of estimators: 150


In [11]:
gb_train_results = evaluate_classifier(
    gradient_boosting,
    X_train,
    y_train,
    "Gradient Boosting - Training",
)

gb_validation_results = evaluate_classifier(
    gradient_boosting,
    X_validation,
    y_validation,
    "Gradient Boosting - Validation",
)


Gradient Boosting - Training results
roc_auc: 0.7501
pr_auc: 0.7944
accuracy: 0.6814
precision: 0.7185
recall: 0.6832
f1: 0.7004

Confusion matrix:
[[5289 2498]
 [2956 6376]]

Gradient Boosting - Validation results
roc_auc: 0.7439
pr_auc: 0.7180
accuracy: 0.6630
precision: 0.5804
recall: 0.7107
f1: 0.6390

Confusion matrix:
[[3642 2153]
 [1212 2978]]


In [12]:
model_comparison_df = pd.DataFrame([
    {
        "model": "Logistic Regression",
        **validation_results,
    },
    {
        "model": "Decision Tree",
        **tree_validation_results,
    },
    {
        "model": "Random Forest",
        **rf_validation_results,
    },
    {
        "model": "Gradient Boosting",
        **gb_validation_results,
    },
])

model_comparison_df = (
    model_comparison_df
    .drop(columns=["dataset"])
    .set_index("model")
    .sort_values("roc_auc", ascending=False)
)

display(model_comparison_df.round(4))

,roc_auc,pr_auc,accuracy,precision,recall,f1
model,,,,,,
Logistic Regression,0.7468,0.7187,0.6646,0.5812,0.7181,0.6425
Gradient Boosting,0.7439,0.7180,0.6630,0.5804,0.7107,0.6390
Random Forest,0.7398,0.7121,0.6554,0.5709,0.7196,0.6367
Decision Tree,0.7338,0.6885,0.6488,0.5641,0.7172,0.6315


In [13]:
# Logistic Regression coefficients
logistic_coefficients = (
    logistic_baseline.named_steps["classifier"].coef_[0]
)

coefficient_df = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": logistic_coefficients,
})

coefficient_df["absolute_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Logistic Regression coefficients:")
display(coefficient_df.round(4))


# Gradient Boosting feature importance
gb_importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": gradient_boosting.feature_importances_,
})

gb_importance_df = gb_importance_df.sort_values(
    "importance",
    ascending=False
)

print("\nGradient Boosting feature importance:")
display(gb_importance_df.round(4))

Logistic Regression coefficients:


,feature,coefficient,absolute_coefficient
1,log1p_frequency,0.6349,0.6349
2,log1p_monetary,0.2525,0.2525
3,log1p_unique_products,0.1716,0.1716
0,recency_days,-0.0815,0.0815
4,log1p_cancellation_value,0.0408,0.0408



Gradient Boosting feature importance:


,feature,importance
1,log1p_frequency,0.5710
2,log1p_monetary,0.2807
3,log1p_unique_products,0.0639
0,recency_days,0.0482
4,log1p_cancellation_value,0.0362


In [14]:
from sklearn.base import clone

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

logistic_tuning_results = []

for C in C_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    train_metrics = evaluate_classifier(
        model,
        X_train,
        y_train,
        f"Logistic Regression C={C} - Training"
    )

    validation_metrics = evaluate_classifier(
        model,
        X_validation,
        y_validation,
        f"Logistic Regression C={C} - Validation"
    )

    logistic_tuning_results.append({
        "C": C,
        "train_roc_auc": train_metrics["roc_auc"],
        "validation_roc_auc": validation_metrics["roc_auc"],
        "train_pr_auc": train_metrics["pr_auc"],
        "validation_pr_auc": validation_metrics["pr_auc"],
        "validation_precision": validation_metrics["precision"],
        "validation_recall": validation_metrics["recall"],
        "validation_f1": validation_metrics["f1"],
    })

logistic_tuning_df = pd.DataFrame(logistic_tuning_results)

print("Logistic Regression hyperparameter comparison:")
display(logistic_tuning_df.round(4))


Logistic Regression C=0.01 - Training results
roc_auc: 0.7267
pr_auc: 0.7732
accuracy: 0.6670
precision: 0.7016
recall: 0.6771
f1: 0.6891

Confusion matrix:
[[5099 2688]
 [3013 6319]]

Logistic Regression C=0.01 - Validation results
roc_auc: 0.7472
pr_auc: 0.7190
accuracy: 0.6661
precision: 0.5826
recall: 0.7203
f1: 0.6442

Confusion matrix:
[[3633 2162]
 [1172 3018]]

Logistic Regression C=0.1 - Training results
roc_auc: 0.7267
pr_auc: 0.7731
accuracy: 0.6661
precision: 0.7014
recall: 0.6747
f1: 0.6878

Confusion matrix:
[[5107 2680]
 [3036 6296]]

Logistic Regression C=0.1 - Validation results
roc_auc: 0.7469
pr_auc: 0.7188
accuracy: 0.6648
precision: 0.5814
recall: 0.7186
f1: 0.6428

Confusion matrix:
[[3627 2168]
 [1179 3011]]

Logistic Regression C=1.0 - Training results
roc_auc: 0.7267
pr_auc: 0.7731
accuracy: 0.6662
precision: 0.7014
recall: 0.6749
f1: 0.6879

Confusion matrix:
[[5106 2681]
 [3034 6298]]

Logistic Regression C=1.0 - Validation results
roc_auc: 0.7468
pr_auc: 0.

,C,train_roc_auc,validation_roc_auc,train_pr_auc,validation_pr_auc,validation_precision,validation_recall,validation_f1
0,0.01,0.7267,0.7472,0.7732,0.7190,0.5826,0.7203,0.6442
1,0.10,0.7267,0.7469,0.7731,0.7188,0.5814,0.7186,0.6428
2,1.00,0.7267,0.7468,0.7731,0.7187,0.5812,0.7181,0.6425
3,10.00,0.7267,0.7468,0.7731,0.7187,0.5812,0.7181,0.6425
4,100.00,0.7267,0.7468,0.7731,0.7187,0.5812,0.7181,0.6425


In [15]:
from sklearn.ensemble import GradientBoostingClassifier

gb_configurations = [
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
]

gb_tuning_results = []

for config in gb_configurations:

    model = GradientBoostingClassifier(
        **config,
        min_samples_leaf=30,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_probabilities = model.predict_proba(X_train)[:, 1]
    validation_probabilities = model.predict_proba(X_validation)[:, 1]

    train_predictions = (train_probabilities >= 0.5).astype(int)
    validation_predictions = (validation_probabilities >= 0.5).astype(int)

    gb_tuning_results.append({
        **config,
        "train_roc_auc": roc_auc_score(y_train, train_probabilities),
        "validation_roc_auc": roc_auc_score(
            y_validation, validation_probabilities
        ),
        "train_pr_auc": average_precision_score(
            y_train, train_probabilities
        ),
        "validation_pr_auc": average_precision_score(
            y_validation, validation_probabilities
        ),
        "validation_precision": precision_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "validation_recall": recall_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "validation_f1": f1_score(
            y_validation, validation_predictions, zero_division=0
        ),
    })

gb_tuning_df = pd.DataFrame(gb_tuning_results)

gb_tuning_df = gb_tuning_df.sort_values(
    "validation_roc_auc",
    ascending=False
)

display(gb_tuning_df.round(4))

,n_estimators,learning_rate,max_depth,train_roc_auc,validation_roc_auc,train_pr_auc,validation_pr_auc,validation_precision,validation_recall,validation_f1
3,100,0.05,2,0.7370,0.7477,0.7834,0.7211,0.5808,0.7165,0.6415
2,200,0.03,2,0.7380,0.7477,0.7845,0.7212,0.5822,0.7136,0.6412
5,200,0.05,2,0.7421,0.7476,0.7880,0.7215,0.5826,0.7115,0.6406
1,150,0.03,2,0.7363,0.7476,0.7828,0.7210,0.5835,0.7146,0.6424
4,150,0.05,2,0.7397,0.7475,0.7860,0.7212,0.5821,0.7124,0.6407
0,100,0.03,2,0.7340,0.7460,0.7788,0.7173,0.5793,0.7198,0.6420
6,100,0.05,3,0.7455,0.7443,0.7908,0.7187,0.5793,0.7095,0.6378
7,150,0.05,3,0.7501,0.7439,0.7944,0.7180,0.5804,0.7107,0.6390
8,200,0.05,3,0.7536,0.7430,0.7981,0.7176,0.5816,0.7103,0.6395


In [16]:
tuned_gradient_boosting = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=30,
    random_state=42
)

tuned_gradient_boosting.fit(X_train, y_train)

print("Tuned Gradient Boosting trained successfully.")

tuned_gb_validation_results = evaluate_classifier(
    tuned_gradient_boosting,
    X_validation,
    y_validation,
    "Tuned Gradient Boosting - Validation"
)

Tuned Gradient Boosting trained successfully.

Tuned Gradient Boosting - Validation results
roc_auc: 0.7476
pr_auc: 0.7215
accuracy: 0.6650
precision: 0.5826
recall: 0.7115
f1: 0.6406

Confusion matrix:
[[3659 2136]
 [1209 2981]]


In [19]:
tuned_logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.01,
        max_iter=1000,
        random_state=42
    ))
])

tuned_logistic.fit(X_train, y_train)

print("Tuned Logistic Regression trained successfully.")

Tuned Logistic Regression trained successfully.


In [20]:
tuned_logistic_validation_results = evaluate_classifier(
    tuned_logistic,
    X_validation,
    y_validation,
    "Tuned Logistic Regression - Validation"
)


Tuned Logistic Regression - Validation results
roc_auc: 0.7472
pr_auc: 0.7190
accuracy: 0.6661
precision: 0.5826
recall: 0.7203
f1: 0.6442

Confusion matrix:
[[3633 2162]
 [1172 3018]]


In [21]:
final_candidate_comparison = pd.DataFrame([
    {
        "model": "Tuned Logistic Regression",
        **tuned_logistic_validation_results,
    },
    {
        "model": "Tuned Gradient Boosting",
        **tuned_gb_validation_results,
    },
])

display(
    final_candidate_comparison
    .set_index("model")
    .round(4)
)

,dataset,roc_auc,pr_auc,accuracy,precision,recall,f1
model,,,,,,,
Tuned Logistic Regression,Tuned Logistic Regression - Validation,0.7472,0.7190,0.6661,0.5826,0.7203,0.6442
Tuned Gradient Boosting,Tuned Gradient Boosting - Validation,0.7476,0.7215,0.6650,0.5826,0.7115,0.6406


In [22]:
thresholds = np.arange(0.20, 0.81, 0.05)

validation_probabilities = tuned_logistic.predict_proba(
    X_validation
)[:, 1]

threshold_results = []

for threshold in thresholds:

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "accuracy": accuracy_score(
            y_validation, predictions
        ),
        "predicted_positive_rate": predictions.mean(),
    })

threshold_results_df = pd.DataFrame(threshold_results)

display(threshold_results_df.round(4))

,threshold,precision,recall,f1,accuracy,predicted_positive_rate
0,0.20,0.4206,0.9986,0.5919,0.4221,0.9963
1,0.25,0.4250,0.9897,0.5946,0.4338,0.9773
2,0.30,0.4370,0.9685,0.6023,0.4633,0.9299
3,0.35,0.4672,0.9205,0.6198,0.5261,0.8268
4,0.40,0.5159,0.8317,0.6368,0.6019,0.6765
5,0.45,0.5507,0.7621,0.6394,0.6393,0.5807
6,0.50,0.5826,0.7203,0.6442,0.6661,0.5188
7,0.55,0.6191,0.6592,0.6385,0.6868,0.4468
8,0.60,0.6708,0.5914,0.6286,0.7068,0.3700
9,0.65,0.7187,0.5208,0.6039,0.7134,0.3041


In [23]:
thresholds_to_compare = [0.40, 0.50, 0.60]

threshold_confusion_results = []

for threshold in thresholds_to_compare:

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions
    ).ravel()

    threshold_confusion_results.append({
        "threshold": threshold,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_validation, predictions, zero_division=0
        ),
    })

threshold_confusion_df = pd.DataFrame(
    threshold_confusion_results
)

display(threshold_confusion_df.round(4))

,threshold,true_negatives,false_positives,false_negatives,true_positives,precision,recall,f1
0,0.4,2525,3270,705,3485,0.5159,0.8317,0.6368
1,0.5,3633,2162,1172,3018,0.5826,0.7203,0.6442
2,0.6,4579,1216,1712,2478,0.6708,0.5914,0.6286


In [24]:
selected_model = tuned_logistic
selected_model_name = "Tuned Logistic Regression"
selected_threshold = 0.50

print("Selected model:", selected_model_name)
print("Selected threshold:", selected_threshold)
print("Selected features:", feature_columns)

Selected model: Tuned Logistic Regression
Selected threshold: 0.5
Selected features: ['recency_days', 'log1p_frequency', 'log1p_monetary', 'log1p_unique_products', 'log1p_cancellation_value']


In [25]:
model_selection_summary = pd.DataFrame([{
    "selected_model": selected_model_name,
    "threshold": selected_threshold,
    "validation_roc_auc": tuned_logistic_validation_results["roc_auc"],
    "validation_pr_auc": tuned_logistic_validation_results["pr_auc"],
    "validation_precision": tuned_logistic_validation_results["precision"],
    "validation_recall": tuned_logistic_validation_results["recall"],
    "validation_f1": tuned_logistic_validation_results["f1"],
    "features": ", ".join(feature_columns),
}])

display(model_selection_summary.round(4))

,selected_model,threshold,validation_roc_auc,validation_pr_auc,validation_precision,validation_recall,validation_f1,features
0,Tuned Logistic Regression,0.5,0.7472,0.719,0.5826,0.7203,0.6442,"recency_days, log1p_frequency, log1p_monetary,..."
